# 05 · pytest：让代码能被信任

**时间预算**：3 小时。

前四课都在"分析数据"。从今天起换一条腿：**写能被人信任的代码**。

为什么现在学它，有两个理由：

1. **它是所有开源贡献的门票。** Airflow（以及任何正经项目）的 PR 都要求带测试，
   没有测试的改动 review 会直接被打回。
2. **它能发现你没想到的情况。** 我今天核对你的 `scripts/` 时，就靠测试找出了两个真实问题——
   而这两个问题肉眼看不出来。

### 这节课怎么用

老规矩，讲一段练一段。不一样的是：**你在给「自己项目里的真实代码」写测试**，
不是玩具例子。做完这节课，你的仓库里会多一个 `tests/` 目录和一个 `pytest.ini`——
**这在作品集里是实打实的加分项**。

## 0. 先准备两个小工具

第一件：让 notebook 能在项目根目录里跑 pytest。

> 为什么要绕一下：pytest 本来是个命令行工具，在 PyCharm 的 Terminal 里敲
> `pytest tests/ -v` 就行。但那样你看不到输出，也不方便反复改。
> 所以这里包一个函数，用**当前内核的解释器**去调它——
> 用 `sys.executable` 而不是直接写 `pytest`，是为了保证用的是你项目虚拟环境里的那个，
> 不会跑到全局 Python 上去。

**顺带记一个 Windows 上的经典坑**（这个函数里已经处理好了）：

`subprocess.run(..., text=True)` 在 Windows 上默认用 **GBK** 去解码子进程的输出。
但 pytest 输出的是 **UTF-8**（因为我们的测试名是中文），于是解码直接崩掉，
`result.stdout` 变成 `None`，下一行 `print(result.stdout[-4000:])` 就报
`TypeError: 'NoneType' object is not subscriptable`。

**解法是两头都指定 UTF-8**：给子进程传 `PYTHONIOENCODING=utf-8`（让它输出 UTF-8），
同时给 `subprocess` 传 `encoding="utf-8"`（让它按 UTF-8 解码）。
在 Windows 上处理任何中文输出，这个组合你都会用到。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run_pytest(args: str = "tests/ -v") -> None:
    """在项目根目录（notebook 的上一级）运行 pytest，并把输出打印出来。"""
    # 下面这行和 encoding 参数是为绕开 Windows 的编码坑，原因见本节末尾的说明
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}

    result = subprocess.run(
        [sys.executable, "-m", "pytest", *args.split()],
        cwd="..",
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )
    print(result.stdout[-4000:])
    if result.stderr.strip():
        print("--- stderr ---")
        print(result.stderr[-1500:])


print("内核的解释器:", sys.executable)
print("pytest 版本:", subprocess.run([sys.executable, "-m", "pytest", "--version"],
                                capture_output=True, text=True).stdout.strip())

## 1. 测试是什么：一个断言而已

先看你项目里这个真实函数——`scripts/download_data.py` 里的 `human()`，
它把字节数变成人看的单位：

In [ ]:
import sys
sys.path.insert(0, "..")

from scripts.download_data import human

for size in (0, 512, 1024, 1536, 1024 ** 2):
    print(f"  human({size:>10,}) = {human(size)}")

print()
print("  ^ 这几个结果，是我一眼能判断对错的吗？——不完全是。")
print("    而且如果我把 human() 改坏了，谁会告诉我？没有人。")

**测试就是把"我用眼睛看"换成"程序替我看"。**

最原始的形态就一行——一个断言：

```python
assert human(1024) == "1KB"
```

- 结果对 → 什么都不发生（安静就是好事）
- 结果错 → 抛 `AssertionError`，告诉你"这里不对"

**为什么这比 print 强**：print 要人看，人看会累、会走神、会自以为看懂了。
而断言可以在两秒钟内跑完几百个，次次不偷懒。

In [ ]:
# 通过的断言：静悄悄
assert human(1024) == "1KB"
print("第一个断言通过了 —— 注意它什么都没打印，安静就是好事")

# 失败的断言：抛异常
try:
    assert human(1024) == "一千KB"
except AssertionError:
    print("第二个断言失败了 —— AssertionError。")
    print("但它没告诉你'为什么'，只说'不对'。下面看 pytest 怎么补上这个信息。")

### 🔹 小练习 1 · 三分钟

自己写 **3 个断言**，检查 `human()` 的输出。

要求：至少有一个断言是**错的**（故意写错），看看会发生什么。

**你应该观察到**：错的断言抛 `AssertionError`，而**断言语句本身**会显示出来——
Python 会把不成立的那句表达式原样打印，这就是最朴素的"失败信息"。

In [ ]:
# 你的代码：写 3 个断言，其中一个是错的

## 2. pytest：把断言组织成测试

如果只有一层断言，自己写个脚本就够了。用一个专门的框架是因为它能干这些事：

| 能力 | 没有 pytest | 有 pytest |
| :--- | :--- | :--- |
| 找测试 | 自己数哪些函数该跑 | 自动找所有 `test_*.py` 里的 `test_*` |
| 跑完汇总 | 自己统计过了几个 | 一行 `5 passed` |
| 失败定位 | 看堆栈自己找 | 直接告诉你哪个函数、哪一行、实际值 vs 期望值 |
| 只重跑失败的 | 做不到 | `--lf`（last failed） |
| 参数化 | 写循环 | `@parametrize` 一行跑十个用例 |
| 临时文件 | 自己建自己删 | `tmp_path` 自动给你一个干净目录 |

**两条命名规则记住就行**：文件名 `test_*.py`，函数名 `test_*`。
不符合的 pytest 不会碰。

### 建 `tests/` 目录和配置文件

`pytest.ini` 里那两行配置解决一个实际问题：

- `pythonpath = .` —— 让测试文件能 `from scripts.download_data import human`。
  没有它，pytest 找不到 `scripts` 这个包（因为测试文件在 `tests/` 子目录里）。
- `testpaths = tests` —— 告诉 pytest 只在这个目录里找测试，不用每次敲路径。

In [ ]:
Path("../tests").mkdir(exist_ok=True)
print("tests 目录准备好了:", Path("../tests").resolve())

In [ ]:
%%writefile ../pytest.ini
[pytest]
pythonpath = .
testpaths = tests
addopts = -v

### 第一个测试文件

下面的单元格会把内容**写成一个真实文件** `tests/test_download_data.py`——
不是写在 notebook 里，是真的落到你项目里。

In [ ]:
%%writefile ../tests/test_download_data.py
"""download_data.py 的测试。"""

from scripts.download_data import human


def test_1024_字节是_1KB():
    assert human(1024) == "1KB"


def test_0_字节是_0B():
    assert human(0) == "0B"


def test_一兆字节是_1MB():
    assert human(1024 ** 2) == "1MB"

In [ ]:
run_pytest("tests/ -v")

看到 `3 passed` 了吗？**注意 pytest 打印的格式**：

```
tests/test_download_data.py::test_1024_字节是_1KB PASSED        [ 33%]
```

这是我最喜欢 pytest 的一点：**测试函数的名字就是文档**。
半年后你回来看这份输出，一眼就知道"这个函数承诺了 1024 字节显示成 1KB"。
所以测试函数名要写成**一句人话**（能用中文就用中文），不要叫 `test_human_1`。

### 🔹 小练习 2 · 三分钟

在 `tests/test_download_data.py` 里**再加两个测试**，然后重跑：

1. 一个通过的
2. **一个故意写错的**（比如断言 `human(1024) == "1KB "`，注意多了个空格）

**你应该观察到**：pytest 的失败信息比裸 `assert` 好得多——它会告诉你
**实际值是什么、期望值是什么**，还有一个小箭头 `assert '1KB' == '1KB '`。

> 这就是框架的价值：**它不是让测试能跑，是让测试失败时你能立刻看懂。**

In [ ]:
# 用 %%writefile 重写整个文件（会覆盖），把两个新测试加进去
# 或者直接在 PyCharm 里编辑那个文件——两种都行

## 3. 三类用例：这一节决定你写的是不是"真测试"

新手写测试只写一种：**正常值**。写了三个测试，全在验证"1KB 变成 1KB"。

**一个函数的测试要覆盖三类用例**：

| 类型 | 是什么 | 例子（对 `human()`） |
| :--- | :--- | :--- |
| **正常值** | 最常见的输入 | `human(512)` → `"512B"` |
| **边界值** | 卡在分界线上的输入 | `human(1023)` / `human(1024)` / `human(1023.9)` |
| **异常输入** | 不该出现、但万一出现的输入 | `human(-5)` / `human(None)` |

**为什么边界值最关键**：几乎所有的 bug 都躲在"刚好大一点/刚好小一点"的地方。
`human(1024)` 是对的，不代表 `human(1023.9)` 也是对的。

### 🔹 小练习 3 · 六分钟 · 这节课最重要的一次练习

给 `human()` 补**边界**和**异常**用例。你可以先**猜**结果再验证，但更快的办法是——
**先写断言，写你「觉得应该」的结果**，然后看 pytest 告诉你什么。

把这几个写成测试：

```python
def test_1023_字节还是_B():
    assert human(1023) == "1023B"

def test_刚好不到1KB的浮点数():
    assert human(1023.9) == "1KB"        # ← 先猜，这个能过吗？

def test_负数怎么办():
    assert human(-5) == "0B"              # ← 你觉得负数该显示什么？
```

**你应该观察到**：后两个**都会失败**。这不是你写错了，是**代码有问题**：

- `human(1023.9)` 实际返回 `"1024B"` —— 它没跨过 1024 的门槛，但显示成了 1024
- `human(-5)` 实际返回 `"-5B"` —— 负数完全没处理

> **这就是测试的意义所在**：这两个问题**肉眼永远看不出来**——
> 你得先想到"1023.9 会怎么样"，才会去试。而"想到"这件事，人不如测试可靠。

In [ ]:
# 你的代码：把三个测试加进 tests/test_download_data.py，然后 run_pytest()

# 写完先别看代码，先看 pytest 报什么。
# 我的结论（这两个失败说明什么）：

## 4. `tmp_path`：测试需要临时文件时怎么办

很多函数要读写文件。测试时不能污染真实目录，也不能依赖"上次跑留下的文件"。

pytest 的做法叫 **fixture**：你只要在函数参数里写上 `tmp_path`，
它就会给你一个**每次运行都是全新、跑完自动清理**的临时目录。

看一个真实例子——`scripts/check_notebooks.py` 里的 `check()` 函数，
它接收一个路径、读 JSON、返回 True/False。

In [ ]:
%%writefile ../tests/test_check_notebooks.py
"""check_notebooks.py 的测试。"""

from scripts.check_notebooks import check


def test_合法_notebook_返回True(tmp_path):
    nb = tmp_path / "good.ipynb"
    nb.write_text('{"cells": [], "nbformat": 4, "nbformat_minor": 5}', encoding="utf-8")
    assert check(nb) is True


def test_坏掉的_json_返回False(tmp_path):
    nb = tmp_path / "broken.ipynb"
    nb.write_text('{ 这不是 json', encoding="utf-8")
    assert check(nb) is False

In [ ]:
run_pytest("tests/test_check_notebooks.py -v")

**注意 `tmp_path` 这件事有多省事**：

- 不用 `mkdir`、不用 `os.remove`
- 每个测试拿到**不同的**目录，互不干扰
- 跑完自动清理，你的项目目录一直是干净的

这就是"用框架"和"自己写脚本"的差距。

### 🔹 小练习 4 · 五分钟

再补两个测试，覆盖 `check()` 的**边界情况**：

1. **空文件**（0 字节）—— 它会返回什么？先猜再跑
2. **文件不存在**—— `check()` 会抛异常还是返回 False？先猜再跑

**提示**：你不用手工创造"文件不存在"的场景，直接传一个 `tmp_path / "根本不存在.ipynb"` 就行。

> 第 2 个问题很有价值：**代码没处理的情况，测试会替你暴露出来。**
> 如果它抛异常了，那说明 `check()` 面对不存在的文件会崩——
> 这是个需要决定的问题（该崩？还是该返回 False？），而不是"随手写个测试就完事"。

In [ ]:
# 你的代码

# 我的发现：

## 5. `@parametrize`：一张表测十种情况

上面的 `human()` 测试，你已经写了六七个，每个都是三行。
这种"同一件事、换几个输入"的重复，`@pytest.mark.parametrize` 一行就解决：

In [ ]:
%%writefile ../tests/test_human_parametrized.py
"""把 human() 的用例改成参数化写法。"""

import pytest

from scripts.download_data import human


@pytest.mark.parametrize(
    "size,expected",
    [
        (0, "0B"),
        (1, "1B"),
        (512, "512B"),
        (1023, "1023B"),
        (1024, "1KB"),
        (1024 ** 2, "1MB"),
        (1024 ** 3, "1GB"),
    ],
)
def test_human(size, expected):
    assert human(size) == expected

In [ ]:
run_pytest("tests/test_human_parametrized.py -v")

注意到输出里的 `[ 14%]`、`[ 28%]`……了吗——**一个函数，七次运行**，而且每次失败都能精确定位到是哪个参数。

**参数化最实用的场景**：你发现一个 bug 修好了，就把那个 bug 的输入**加进这张表**，
以后它再也不会悄悄回来（这叫"回归测试"）。

### 🔹 小练习 5 · 四分钟

把刚才发现的**两个问题**写进参数化表里（就加两行）：

```python
(1023.9, "1KB"),    # 现在会失败
(-5, "0B"),         # 现在会失败
```

跑一遍，确认**只有这两行失败**，其他七行还是通过的。

**你应该观察到**：pytest 只报了那两个参数组合失败，其他照样 pass。
**这就是"一张表"的价值**——你能一眼看出"哪几种输入是坏的"，而不用逐个试。

In [ ]:
# 你的代码

## 6. `pytest.raises`：测试"应该报错"的情况

有些输入，正确的行为就是**抛出异常**。这种"期望它出错"的测试这样写：

```python
with pytest.raises(ValueError):
    some_function("坏输入")
```

它的意思是：**接下来这段代码必须抛 ValueError；如果不抛，这个测试就失败。**

In [ ]:
%%writefile ../tests/test_raises_demo.py
"""演示 pytest.raises 的两种用法。"""

import pytest


def test_除以零应该报错():
    with pytest.raises(ZeroDivisionError):
        1 / 0


def test_转整数失败应该报错():
    with pytest.raises(ValueError):
        int("这不是数字")


def test_还能顺便检查报错信息():
    with pytest.raises(KeyError, match="missing_key"):
        {}["missing_key"]

In [ ]:
run_pytest("tests/test_raises_demo.py -v")

### 🔹 小练习 6 · 四分钟

给 `human()` 写一个**"期望报错"的测试**。

问题是：**`human(None)` 现在会怎样？** 先跑一下看，然后：

- 如果它抛了 `TypeError` → 写一个 `pytest.raises(TypeError)` 把它**固定下来**
- 如果它悄悄返回了某个值 → 那就说明它没做输入校验，写个测试把它**记录**下来，
  然后想一下：这算不算问题？

> **这道题没有标准答案**，我要看的是你的判断：面对一个没有校验的函数，
> 你会选择"接受现状、用测试记录下来"，还是"改代码加上校验"？两种都对，
> 但**必须是有意识的选择**，而不是"没想过"。

In [ ]:
# 你的代码

# 我的判断：

## 7. 覆盖率：知道"哪里还没测"

`pytest-cov` 会告诉你**代码里哪些行被测试跑到了、哪些没有**。

注意它的定位：**覆盖率不是目标，是手电筒。**

- ❌ "我要把覆盖率刷到 100%" → 会写出很多没意义的测试
- ✅ "哪一行没被覆盖？为什么？那里会不会藏着我没想到的情况？" → 这才是用法

In [ ]:
run_pytest("tests/ --cov=scripts --cov-report=term-missing")

看输出里 `Missing` 那一列——那是一串行号，表示**这些行一次都没被执行过**。

对 `download_data.py` 来说，大概率会看到 `fetch`、`download`、`main` 这些函数整块没被覆盖，
因为它们要联网、要下载，不好在单元测试里跑。

**这是正常的，也是可以接受的**。你要学会的是**区分两类没覆盖**：

| 情况 | 怎么办 |
| :--- | :--- |
| 依赖网络的函数（`fetch`） | 单元测试不测它，合理。以后学 mock 再说 |
| **有逻辑分支却没测** | ⚠️ **这才是要补的**——每个 `if` 都要有测试走到两边 |

### 🔹 小练习 7 · 五分钟

1. 看一下 `human()` 那个函数（在 `scripts/download_data.py` 里），它有一个 `for` 循环和一个 `if`。
   **问自己：`human()` 里有没有哪一行，现有的测试从来没走到过？**
2. 再看输出里的覆盖率百分比——`check_notebooks.py` 是多少？`data_check.py` 呢？
   **哪个脚本的测试最少？**

**你应该观察到**：`data_check.py` 的覆盖率应该最低（我们还没给它写任何测试），
而 `human()` 的 `return f"{size:.0f}TB"` 那一行（超出 GB 的情况）大概率没被覆盖。

> 想覆盖那一行，加一个测试：`assert human(1024 ** 4) == "1TB"`。
> 这就是"用覆盖率当手电筒"的完整动作。

In [ ]:
# 你的代码

# 我的发现：

## 8. 大练习：给三个脚本都配上测试

**目标**：`data_check.py` 目前是测试最少的一个，这题主要就是补它。

### 任务

在 `tests/` 下新建 `test_data_check.py`，给 `data_check.py` 的函数写测试：

1. **`check_missing()`** —— 用一个你**手工造的小 DataFrame** 喂它
   （提示：`pd.DataFrame({"a": [1, None, 3], "b": ["x", "y", None]})`）
2. **`check_shape()`** —— 同样用手造的小数据
3. 至少写**一个边界用例**：比如"完全没有缺失值的 DataFrame"
4. 至少写**一个参数化测试**

### 要求

- 测试函数名用**中文人话**（`test_没有缺失值时说什么`）
- 跑 `run_pytest("tests/ --cov=scripts --cov-report=term-missing")`，看覆盖率变化
- **写下三个回答**：
  1. 写测试时，你发现 `data_check.py` 里哪个行为是"我之前没意识到"的？
  2. 哪些函数**不值得**写单元测试？为什么？
  3. 这一课下来，你觉得测试最大的作用是什么？

> **别追求覆盖率**。`main()` 这种"把所有函数串起来"的函数，测它性价比很低——
> 它里面全是调用，逻辑都在被调用的函数里。**测那些有判断、有计算的地方。**

### 做完之后

把 `tests/`、`pytest.ini` 和这三个脚本一起提交——
**这一步做完，你的仓库就从一个"练习本"变成了一个"有测试的工程"**。

In [ ]:
# 你的代码

# 1. 我发现的行为：
# 2. 不值得测的函数：
# 3. 测试最大的作用：

## 9. 总结卡片

1. **测试 = 程序替你看结果**。一行 `assert` 就是最小形态，安静通过、失败抛异常。
2. **命名规则**：文件 `test_*.py`、函数 `test_*`。**函数名写成人话——它就是文档。**
3. **三类用例**：正常值、边界值、异常输入。**只写正常值等于没测。**
4. **`tmp_path`** 解决临时文件，**`@parametrize`** 解决重复用例，
   **`pytest.raises`** 测"应该报错"的情况。
5. **覆盖率是手电筒，不是目标**。看 `Missing` 列找"没走到的分支"，
   不要为了刷数字写没意义的测试。
6. **测试最有价值的时刻，是你第一次发现"原来这里会这样"。**
   今天已经发生两次了（`human(1023.9)` 和 `human(-5)`）——那种意外感就是收获。

---

### 下一课预告

今天你学会了"验证代码对不对"。下一课解决另一半问题：**"代码写得规不规范"**。

`ruff` 管风格和常见错误，`mypy` 管类型。它们和测试一样，都是 **CI 会卡你的东西**——
而且它们比测试更快见效：跑一遍，几十个问题一次性摆在你面前。

## 10. 卡住时的提示

**小练习 1**：直接写 `assert human(512) == "512B"` 这种。写错了会看到 `AssertionError`。

**小练习 2**：用 `%%writefile ../tests/test_download_data.py` 重写整个文件
（注意是**覆盖**，要把原来的三个测试也一起写进去）。

**小练习 3**：三个测试直接照抄题目里的代码加进文件，然后 `run_pytest()`。
后两个会失败——**别急着"修"，先读失败信息**，看清楚实际值和期望值分别是什么。

**小练习 4**：
```python
def test_空文件(tmp_path):
    f = tmp_path / "empty.ipynb"
    f.write_text("", encoding="utf-8")
    assert check(f) is False

def test_文件不存在(tmp_path):
    assert check(tmp_path / "没有这个文件.ipynb") is False
```
第二个大概率会失败（抛 `FileNotFoundError`）。**这个失败是有价值的**——它告诉你
`check()` 没处理"文件不存在"的情况。

**小练习 5**：在参数化列表里加两行就行。注意**加了之后别删**——
它们现在"已知会失败"，以后修好代码它们就会自动变成"已修好"的证据。

**小练习 6**：先跑 `print(human(None))` 看结果，再决定写哪种测试。

**小练习 7**：翻到 `scripts/download_data.py` 看 `human()` 的代码结构——
一个 for 循环里有一个 if，循环外有一个 return。**问自己：那个循环外的 return 什么时候执行？**

**大练习**：`data_check.py` 里的函数都是"接收一个 DataFrame，打印一堆东西，返回 None"。
这种函数**不好断言输出**，因为输出是打印出来的。两个办法：
① 用一个只有两三行的小 DataFrame 跑一遍，只要**不报错**就算过（最省事，也合理）；
② 用 `capsys` fixture 抓打印内容（进阶，这次不强求）。
**先用办法 ①**——测"函数能不能不出错地跑完"，本身就有价值。